# LSTM (PyTorch) - AG News Classification


In [2]:
import os
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import mlflow
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from mlflow_config import configure_mlflow

configure_mlflow()
mlflow.set_experiment("AG_News_Classification")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
os.makedirs('Figures', exist_ok=True)
os.makedirs('results', exist_ok=True)


cuda
NVIDIA A10


In [3]:
train_df = pd.read_csv('data/train.csv')
test_df = pd.read_csv('data/test.csv')
train_df['text'] = train_df['Title'] + ' ' + train_df['Description']
test_df['text'] = test_df['Title'] + ' ' + test_df['Description']
X_train, y_train = train_df['text'].values, (train_df['Class Index']-1).values
X_test, y_test = test_df['text'].values, (test_df['Class Index']-1).values


In [4]:
max_words = 10000
max_len = 150

def tokenize(texts):
    tokens = []
    for t in texts:
        tokens.append(str(t).lower().split())
    return tokens

train_tokens = tokenize(X_train)
counter = Counter([w for s in train_tokens for w in s])
vocab = {w:i+2 for i,(w,_) in enumerate(counter.most_common(max_words-2))}
vocab['<PAD>']=0
vocab['<UNK>']=1
vocab_size = len(vocab)

def encode(tokens):
    seqs = []
    for s in tokens:
        seq = [vocab.get(w,1) for w in s][:max_len]
        seq += [0]*(max_len-len(seq))
        seqs.append(seq)
    return torch.tensor(seqs, dtype=torch.long)

X_train_seq = encode(train_tokens)
X_test_seq = encode(tokenize(X_test))
y_train_t = torch.tensor(y_train, dtype=torch.long)
y_test_t = torch.tensor(y_test, dtype=torch.long)


In [5]:
class NewsDataset(Dataset):
    def __init__(self,x,y):
        self.x=x; self.y=y
    def __len__(self): return len(self.y)
    def __getitem__(self,i): return self.x[i], self.y[i]

train_loader = DataLoader(NewsDataset(X_train_seq,y_train_t), batch_size=64, shuffle=True)
test_loader = DataLoader(NewsDataset(X_test_seq,y_test_t), batch_size=64)


In [6]:
class LSTMClassifier(nn.Module):
    def __init__(self,vocab_size,embed_dim=128,hidden_dim=256,num_layers=1):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers=num_layers, batch_first=True, dropout=0.2 if num_layers>1 else 0)
        self.dropout = nn.Dropout(0.5)
        self.fc1 = nn.Linear(hidden_dim, 128)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(128, 4)
    def forward(self,x):
        x = self.embed(x)
        _, (h_n, _) = self.lstm(x)
        h = h_n[-1]
        h = self.dropout(h)
        h = self.relu(self.fc1(h))
        h = self.dropout(h)
        return self.fc2(h)

model = LSTMClassifier(vocab_size).to(device)
print(model)


LSTMClassifier(
  (embed): Embedding(10000, 128, padding_idx=0)
  (lstm): LSTM(128, 256, batch_first=True)
  (dropout): Dropout(p=0.5, inplace=False)
  (fc1): Linear(in_features=256, out_features=128, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=128, out_features=4, bias=True)
)


In [7]:
with mlflow.start_run(run_name='LSTM_PyTorch'):
    mlflow.log_param('model_type','LSTM_PyTorch')
    mlflow.log_param('max_words',max_words)
    mlflow.log_param('max_len',max_len)
    mlflow.log_param('vocab_size',vocab_size)
    mlflow.log_param('device',str(device))
    
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    
    train_accs, val_accs = [], []
    for epoch in range(10):
        model.train()
        correct,total,loss_sum=0,0,0
        for xb,yb in train_loader:
            xb,yb=xb.to(device),yb.to(device)
            optimizer.zero_grad()
            out=model(xb)
            loss=criterion(out,yb)
            loss.backward()
            optimizer.step()
            loss_sum+=loss.item()
            pred=out.argmax(1)
            correct+=(pred==yb).sum().item()
            total+=yb.size(0)
        train_accs.append(correct/total)
        
        model.eval()
        correct,total=0,0
        with torch.no_grad():
            for xb,yb in test_loader:
                xb,yb=xb.to(device),yb.to(device)
                out=model(xb)
                pred=out.argmax(1)
                correct+=(pred==yb).sum().item()
                total+=yb.size(0)
        val_accs.append(correct/total)
        mlflow.log_metric('val_acc',val_accs[-1],step=epoch)
        print(f'Epoch {epoch+1}: train_acc={train_accs[-1]:.4f} val_acc={val_accs[-1]:.4f}')
    
    model.eval()
    y_pred=[]
    with torch.no_grad():
        for xb,yb in test_loader:
            xb=xb.to(device)
            out=model(xb)
            y_pred.extend(out.argmax(1).cpu().numpy())
    acc=accuracy_score(y_test,y_pred)
    mlflow.log_metric('test_accuracy',acc)
    
    report=classification_report(y_test,y_pred,target_names=['World','Sports','Business','Sci/Tech'])
    with open('results/LSTM_PyTorch_results.txt','w') as f:
        f.write(report)
        f.write(f'\nAccuracy: {acc}\n')
    mlflow.log_artifact('results/LSTM_PyTorch_results.txt')
    
    plt.figure(figsize=(10,4))
    plt.plot(train_accs,label='train')
    plt.plot(val_accs,label='val')
    plt.title('LSTM_PyTorch Accuracy')
    plt.legend()
    plt.savefig('Figures/LSTM_PyTorch_accuracy_loss.png')
    plt.close()
    mlflow.log_artifact('Figures/LSTM_PyTorch_accuracy_loss.png')
    
    cm=confusion_matrix(y_test,y_pred)
    plt.figure(figsize=(8,6))
    sns.heatmap(cm,annot=True,fmt='d',cmap='Blues',xticklabels=['World','Sports','Business','Sci/Tech'],yticklabels=['World','Sports','Business','Sci/Tech'])
    plt.title('LSTM_PyTorch Confusion Matrix')
    plt.savefig('Figures/LSTM_PyTorch_confusion_matrix.png')
    plt.close()
    mlflow.log_artifact('Figures/LSTM_PyTorch_confusion_matrix.png')
    print(report)


    # Save model
    os.makedirs('models', exist_ok=True)
    torch.save({
        'model_state_dict': model.state_dict(),
        'vocab': vocab,
        'max_len': max_len,
        'vocab_size': vocab_size,
    }, 'models/LSTM_PyTorch.pt')
    mlflow.log_artifact('models/LSTM_PyTorch.pt')
    print('Saved models/LSTM_PyTorch.pt')


Epoch 1: train_acc=0.2498 val_acc=0.2500
Epoch 2: train_acc=0.2508 val_acc=0.2500
Epoch 3: train_acc=0.2500 val_acc=0.2500
Epoch 4: train_acc=0.2477 val_acc=0.2500
Epoch 5: train_acc=0.2504 val_acc=0.2500
Epoch 6: train_acc=0.2496 val_acc=0.2501
Epoch 7: train_acc=0.2506 val_acc=0.2530
Epoch 8: train_acc=0.5756 val_acc=0.8438
Epoch 9: train_acc=0.8756 val_acc=0.8864
Epoch 10: train_acc=0.9081 val_acc=0.8983
              precision    recall  f1-score   support

       World       0.92      0.88      0.90      1900
      Sports       0.93      0.98      0.96      1900
    Business       0.87      0.86      0.86      1900
    Sci/Tech       0.88      0.87      0.88      1900

    accuracy                           0.90      7600
   macro avg       0.90      0.90      0.90      7600
weighted avg       0.90      0.90      0.90      7600

Saved models/LSTM_PyTorch.pt
